# WaveNet Colab Notebook

In [ ]:
!pip install "sympy==1.13.3"

In [ ]:
#Import files from Google Drive (add your own paths)
import os
from google.colab import drive
drive.mount("/content/gdrive", force_remount=True)
os.chdir('/content/gdrive/MyDrive/ColabNotebooks')
print(os.getcwd())

In [ ]:
#Create snapshot directory
snapshot_dir = '/content/gdrive/MyDrive/ColabNotebooks/wavenet_snapshots'
os.makedirs(snapshot_dir, exist_ok=True)
print(f"Snapshots will be saved to: {snapshot_dir}")

In [ ]:
!unzip /content/gdrive/MyDrive/ColabNotebooks/guitar_dataset.zip -d /content/guitar_dataset

In [ ]:
import torch
import torch.optim as optim
import torch.nn.functional as F
from torch.autograd import Variable
import numpy as np

from wavenet_model import WaveNetModel
from wavenet_training import WavenetTrainer
from paired_audio_data import PairedWavenetDataset
from model_logging import TensorboardLogger

print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

logger = TensorboardLogger(log_dir='logs')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Initialize Model
model = WaveNetModel(
    layers=8,
    blocks=3,
    dilation_channels=64,
    residual_channels=64,
    skip_channels=512,
    end_channels=256,
    mu_law_classes=256,
    output_length=64,
    bias=True,
    device=device
)


print('Model initialized:')
print(model)
print('Receptive Field: ', model.receptive_field)
print(hasattr(model, 'mu_law_classes'))
print(model.mu_law_classes)

In [ ]:
# Initialize Dataset
data = PairedWavenetDataset(

    dataset_file='/content/guitar_dataset/guitar_paired_dataset.npz',
    item_length=model.receptive_field + model.output_length - 1,
    target_length=model.output_length,
    clean_dir='/content/guitar_dataset/clean_guitar',
    processed_dir='/content/guitar_dataset/overdrive_guitar',
    sampling_rate=16000,
    mono=True,
    test_stride=100
)

print('Dataset initialized:')
print(data)
print(len(data))


In [ ]:
# Initialize Logger and Trainer
logger = TensorboardLogger(log_interval=100, validation_interval=200)


trainer = WavenetTrainer(model=model,
                         dataset=data,
                         lr = 0.0001,
                         gradient_clipping=1.0,
                         snapshot_path=snapshot_dir,
                         snapshot_name='wavenet_snapshots',
                         snapshot_interval=1000,
                         logger=logger,
                         device=device)

logger.trainer = trainer

print('start training...')
trainer.train(batch_size=128, epochs=10)

In [ ]:
# Restore Model State and Resume Training
from torch.optim.lr_scheduler import ReduceLROnPlateau
logger = TensorboardLogger(log_interval=300, validation_interval=400)

trainer = WavenetTrainer(model=model,
                         dataset=data,
                         lr = 0.0001,
                         gradient_clipping=1.0,
                         snapshot_path=snapshot_dir,
                         snapshot_name='wavenet_snapshots',
                         snapshot_interval=1000,
                         logger=logger,
                         device=device)

logger.trainer = trainer
model = trainer.model
optimizer = trainer.optimizer
scheduler = trainer.scheduler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = '/content/gdrive/MyDrive/ColabNotebooks/wavenet_snapshots/wavenet_snapshots_best.pth' 
checkpoint = torch.load(checkpoint_path, map_location=device)

# load checkpoint states from wavenet_snapshots_best.pth
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

start_epoch = checkpoint['epoch'] + 1
start_step = checkpoint['step']


model.train()

print(f"Checkpoint loaded successfully. Continuing training from Epoch {start_epoch}, Step {start_step}.")
print('\n start training...')
trainer.train(batch_size=128,
              epochs=55,
              start_epoch=start_epoch,
              continue_training_at_step=start_step)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = '/content/gdrive/MyDrive/ColabNotebooks/wavenet_snapshots/wavenet_snapshots_best.pth'

ckpt = torch.load(model_path, map_location=device)

model.load_state_dict(ckpt['model_state_dict'])

model = model.to(device)
model.eval()

print("Model restored from checkpoint and set to eval mode.")

Model restored from checkpoint and set to eval mode.


In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display
from audio_utils import quantize_data, mu_law_expansion


CLEAN_AUDIO_PATH = '/content/gdrive/MyDrive/ColabNotebooks/2-0.wav'
OUTPUT_PATH = '/content/gdrive/MyDrive/ColabNotebooks/generated_audio/processed_2-0.wav'
SAMPLING_RATE = 16000

clean_y, sr = librosa.load(CLEAN_AUDIO_PATH, sr=16000, mono=True)
print(f"Clean audio length: {len(clean_y)} samples")



q_clean = quantize_data(clean_y, 256)
clean_tensor = torch.from_numpy(q_clean).float().to(device)
clean_one_hot = torch.zeros(256, clean_tensor.size(0), device=device)
clean_one_hot.scatter_(0, clean_tensor.unsqueeze(0).long(), 1.)
clean_one_hot = clean_one_hot.unsqueeze(0)


output_segments = []
chunk_size = model.receptive_field + model.output_length
for start in range(0, clean_one_hot.size(2) - chunk_size, model.output_length):
    segment = clean_one_hot[:, :, start:start + chunk_size]
    with torch.no_grad():
        y_pred = model(segment)
    y_pred = y_pred.reshape(-1, 256)
    probabilities = torch.softmax(y_pred / 0.5, dim=1)
    predicted = torch.multinomial(probabilities, num_samples=1).squeeze(1)
    output_segments.append(predicted.cpu().numpy())

predicted_indices = np.concatenate(output_segments)
decoded_audio = mu_law_expansion(predicted_indices)

sf.write(OUTPUT_PATH, decoded_audio, SAMPLING_RATE)
print("\n--- DEBUG INFO ---")
print(f"Clean Audio Length (samples): {len(clean_y)}")
print(f"Processed Audio Length (samples): {len(decoded_audio)}")
print(f"Processed Audio Duration (seconds): {len(decoded_audio)/SAMPLING_RATE:.2f}")
print(f"Output Range: {np.min(decoded_audio):.4f} to {np.max(decoded_audio):.4f}")
plt.figure(figsize=(14,5))
librosa.display.waveshow(decoded_audio, sr=SAMPLING_RATE, color='blue')
plt.ylim(-1.0, 1.0)
plt.title("Generated Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()
display(Audio(decoded_audio, rate=SAMPLING_RATE))
print(" Saved:", OUTPUT_PATH)